# Retail ETL pipeline

Run the setup cell once, then call `etl.run_pipeline()` to transform data staged by `extract.sql` and publish it to the `clean` schema.

In [ ]:
%load_ext sql
from pathlib import Path
from sqlalchemy import create_engine

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / 'etl' / 'pipeline.sql').is_file()
)
database_url = (project_root / 'secrets' / 'database_url.txt').read_text().strip()
engine = create_engine(database_url)
%sql engine

In [ ]:
# Install or update the SQL function. Run this once per database.
from sqlalchemy import text

pipeline_sql = (project_root / 'etl' / 'pipeline.sql').read_text()
pipeline_sql = '\n'.join(
    line for line in pipeline_sql.splitlines() if not line.startswith('\\')
)
with engine.begin() as connection:
    connection.execute(text(pipeline_sql))

In [ ]:
%%sql
SELECT etl.run_pipeline();

In [ ]:
%%sql
SELECT category, count(*) AS rows
FROM clean.fact_sales
GROUP BY category
ORDER BY rows DESC;